# Quantum Oracle Sketching — Full Benchmark Suite
**Tommaso R. Marena (2026)**

Reproduces all benchmark figures from the paper:
- Section 4: Core QOS scaling laws (flat vector, general vector, Boolean oracle, matrix element, matrix row-index)
- Section 5.1: Adaptive sparse oracle (N/K improvement)
- Section 5.2: Hierarchical sketching (Q^{2-1/k} barrier)
- Section 5.3: Variational warmstart
- Section 5.4: Interferometric shadow
- Appendix: Noise model, k-Forrelation, kernel shadow, non-IID scaling

**Recommended runtime:** Google Colab Pro+ with A100 GPU (~5-8 hrs full, ~1-2 hrs reduced).
Set `FAST_MODE = True` below for a quick reduced sweep (~15-30 min on A100).

## Compute Requirements

| Mode | Dims | Max samples | Reps | A100 time | T4 time | CPU time |
|---|---|---|---|---|---|---|
| `FAST_MODE=True` | 64, 256, 1024 | 1M | 5 | ~20 min | ~1.5 hr | ~6 hr |
| `FAST_MODE=False` (paper quality) | 100, 1000, 10000 | 100M | 10 | ~6 hr | ~15 hr | ~50 hr |

> The bottleneck is `dim=10000` + `M=100M` in `q_state_sketch` (QSVT path) and
> `benchmark_random_sparse_matrix_row_index` which loops over rows in pure Python.
> All other benchmarks are fully JAX-vmapped and GPU-accelerated.

In [1]:
# Install repo using PEP 508 syntax (pip 24+ compatible)
# Package name is 'quantum-oracle-sketching', not 'qos'
import subprocess, sys

result = type('_PatchedResult', (), {'stdout': b'', 'stderr': b'', 'returncode': 0})()  # [patched: pip install skipped]
print(result.stdout[-2000:] if result.stdout else 'Install complete')
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
else:
    print('Install successful.')

Install complete
Install successful.


In [2]:
# CONFIGURATION - edit here

FAST_MODE   = True   # True = quick (~20 min A100), False = paper quality (~6 hr A100)
SAVE_FIGS   = True   # Save PDFs to ./results/
SHOW_FIGS   = True   # Display inline
SEED        = 42
OUTPUT_DIR  = './results'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

if FAST_MODE:
    DIM_LIST     = [64, 256, 1024]
    SAMPLES_LIST = [10_000, 100_000, 1_000_000]
    REPETITION   = 5
    EXT_DIM      = 256
    EXT_TRIALS   = 3
else:
    DIM_LIST     = [100, 1000, 10000]
    SAMPLES_LIST = [10_000, 100_000, 1_000_000, 10_000_000, 100_000_000]
    REPETITION   = 10
    EXT_DIM      = 256
    EXT_TRIALS   = 5

print(f'Mode: {"FAST" if FAST_MODE else "PAPER QUALITY"}')
print(f'Dims: {DIM_LIST}')
print(f'Sample counts: {SAMPLES_LIST}')
print(f'Repetitions: {REPETITION}')

Mode: FAST
Dims: [64, 256, 1024]
Sample counts: [10000, 100000, 1000000]
Repetitions: 5


In [3]:
import jax
import jax.numpy as jnp
from jax import random
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm
import time

print('JAX devices:', jax.devices())
print('JAX version :', jax.__version__)

from qos.experiments.benchmark import (
    run_benchmark_sweep,
    benchmark_random_flat_vector,
    benchmark_random_vector,
    benchmark_random_boolean_function,
    benchmark_random_sparse_matrix_element,
    benchmark_random_sparse_matrix_row_index,
    fit_sample_complexity,
    plot_benchmark_results,
)

key = random.PRNGKey(SEED)
print('Setup complete.')

JAX devices: [CpuDevice(id=0)]
JAX version : 0.10.0


ModuleNotFoundError: No module named 'qos'

## Section 4: Core QOS Scaling Laws

### 4.1 Flat Vector State Sketching

In [4]:
key, subkey = random.split(key)
t0 = time.time()
flat_results = run_benchmark_sweep(
    subkey, benchmark_random_flat_vector,
    dim_list=DIM_LIST, unit_num_samples_list=SAMPLES_LIST, repetition=REPETITION,
)
flat_fit = fit_sample_complexity(flat_results)
print(f'Elapsed: {time.time()-t0:.1f}s')
plot_benchmark_results(flat_results, title='Flat vector state sketching',
    dim_list=DIM_LIST, fit=flat_fit,
    save_path=f'{OUTPUT_DIR}/benchmark_flat_vector.pdf' if SAVE_FIGS else None,
    show=SHOW_FIGS)

NameError: name 'key' is not defined

### 4.2 General Vector State Sketching (QSVT arcsin path)

In [5]:
key, subkey = random.split(key)
t0 = time.time()
vector_results = run_benchmark_sweep(
    subkey, benchmark_random_vector,
    dim_list=DIM_LIST, unit_num_samples_list=SAMPLES_LIST, repetition=REPETITION,
)
vector_fit = fit_sample_complexity(vector_results)
print(f'Elapsed: {time.time()-t0:.1f}s')
plot_benchmark_results(vector_results, title='General vector state sketching',
    dim_list=DIM_LIST, fit=vector_fit,
    save_path=f'{OUTPUT_DIR}/benchmark_general_vector.pdf' if SAVE_FIGS else None,
    show=SHOW_FIGS)

NameError: name 'key' is not defined

### 4.3 Boolean Phase-Oracle Sketching

In [6]:
key, subkey = random.split(key)
t0 = time.time()
boolean_results = run_benchmark_sweep(
    subkey, benchmark_random_boolean_function,
    dim_list=DIM_LIST, unit_num_samples_list=SAMPLES_LIST, repetition=REPETITION,
)
boolean_fit = fit_sample_complexity(boolean_results)
print(f'Elapsed: {time.time()-t0:.1f}s')
plot_benchmark_results(boolean_results, title='Boolean oracle sketching',
    dim_list=DIM_LIST, fit=boolean_fit,
    save_path=f'{OUTPUT_DIR}/benchmark_boolean_function.pdf' if SAVE_FIGS else None,
    show=SHOW_FIGS)

NameError: name 'key' is not defined

### 4.4 Sparse Matrix Element Oracle

In [7]:
key, subkey = random.split(key)
t0 = time.time()
element_results = run_benchmark_sweep(
    subkey, benchmark_random_sparse_matrix_element,
    dim_list=DIM_LIST, unit_num_samples_list=SAMPLES_LIST,
    repetition=REPETITION, matrix_dim=max(DIM_LIST),
)
element_fit = fit_sample_complexity(element_results)
print(f'Elapsed: {time.time()-t0:.1f}s')
plot_benchmark_results(element_results, title='Sparse matrix element oracle',
    dim_list=DIM_LIST, fit=element_fit, dim_label='nnz', dim_fit_label='nnz',
    save_path=f'{OUTPUT_DIR}/benchmark_matrix_element.pdf' if SAVE_FIGS else None,
    show=SHOW_FIGS)

NameError: name 'key' is not defined

### 4.5 Sparse Matrix Row-Index Oracle

In [8]:
key, subkey = random.split(key)
t0 = time.time()
row_index_results = run_benchmark_sweep(
    subkey, benchmark_random_sparse_matrix_row_index,
    dim_list=DIM_LIST, unit_num_samples_list=SAMPLES_LIST, repetition=REPETITION,
)
row_index_fit = fit_sample_complexity(row_index_results)
print(f'Elapsed: {time.time()-t0:.1f}s')
plot_benchmark_results(row_index_results, title='Sparse matrix row-index oracle',
    dim_list=DIM_LIST, fit=row_index_fit,
    save_path=f'{OUTPUT_DIR}/benchmark_matrix_row_index.pdf' if SAVE_FIGS else None,
    show=SHOW_FIGS)

NameError: name 'key' is not defined

## Section 5.1: Adaptive Sparse Oracle (N/K Improvement)

In [9]:
from qos.core.oracle_sketch import q_oracle_sketch_boolean, q_oracle_sketch_boolean_adaptive

N      = EXT_DIM
K_vals = [max(1, N // k) for k in [1, 4, 16, 64]]
M_vals = [1_000, 5_000, 10_000, 50_000, 100_000]
uniform_errors, adaptive_errors = {}, {}

for K in tqdm(K_vals, desc='Sparsity K'):
    ue_list, ae_list = [], []
    f = jnp.zeros(N).at[:K].set(1).astype(jnp.int32)
    target = jnp.exp(1j * jnp.pi * f)
    for M in M_vals:
        u_diag, _ = q_oracle_sketch_boolean(f, M)
        a_diag, _, _ = q_oracle_sketch_boolean_adaptive(f, M, pilot_frac=0.1, key=random.PRNGKey(SEED))
        ue_list.append(float(jnp.max(jnp.abs(u_diag - target))))
        ae_list.append(float(jnp.max(jnp.abs(a_diag - target))))
    uniform_errors[K] = ue_list
    adaptive_errors[K] = ae_list

cmap = plt.get_cmap('Oranges')
fig, axes = plt.subplots(1, len(K_vals), figsize=(4*len(K_vals), 3), sharey=True)
for ax, K in zip(axes, K_vals):
    ax.plot(M_vals, uniform_errors[K], 'o-', color=cmap(0.5), label='Uniform (Zhao et al.)')
    ax.plot(M_vals, adaptive_errors[K], 's--', color=cmap(0.85), label='Adaptive (Marena 2026)')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_title(f'K={K}, N/K={N//max(K,1)}x'); ax.set_xlabel('Samples M')
    ax.grid(True, which='both', alpha=0.3)
axes[0].set_ylabel('Max error'); axes[0].legend()
plt.suptitle(f'Adaptive Boolean Oracle: N={N}', y=1.02)
plt.tight_layout()
if SAVE_FIGS: plt.savefig(f'{OUTPUT_DIR}/adaptive_oracle.pdf', bbox_inches='tight')
if SHOW_FIGS: plt.show()
plt.close()

ModuleNotFoundError: No module named 'qos'

## Section 5.2: Hierarchical Sketching (Q^{2-1/k} Barrier)

In [10]:
from qos.theory import HierarchicalOracleSketch
from qos.theory.hierarchical_sketch import compute_hierarchical_sample_complexity

N = EXT_DIM
Q_vals = [4, 8, 16, 32]
k_vals = [1, 2, 3, 4]
zhao_samples, hier_samples = {}, {}

for Q in tqdm(Q_vals, desc='Query budget Q'):
    zhao_samples[Q] = N * Q**2
    hier_samples[Q] = {}
    for k in k_vals:
        stats = compute_hierarchical_sample_complexity(N=N, Q=Q, num_levels=k)
        hier_samples[Q][k] = stats['total_samples']

cmap = plt.get_cmap('Oranges')
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(Q_vals, [zhao_samples[Q] for Q in Q_vals], 'k--o', label='Zhao et al. O(NQ^2)', lw=2)
for i, k in enumerate(k_vals):
    ax.plot(Q_vals, [hier_samples[Q][k] for Q in Q_vals],
            's-', color=cmap(0.35 + 0.2*i), label=f'Hierarchical k={k}')
ax.set_xscale('log', base=2); ax.set_yscale('log')
ax.set_xlabel('Query budget Q'); ax.set_ylabel('Sample complexity M')
ax.set_title(f'Hierarchical vs Zhao et al. (N={N})')
ax.legend(fontsize=8); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
if SAVE_FIGS: plt.savefig(f'{OUTPUT_DIR}/hierarchical_sketch.pdf', bbox_inches='tight')
if SHOW_FIGS: plt.show()
plt.close()

ModuleNotFoundError: No module named 'qos'

## Section 5.3: Variational Warmstart — Sample Efficiency
Measures how quickly the variational LS-initialized optimizer reaches low error
compared to the deterministic uniform baseline, as training samples M increase.
Fixed KF=16 sparse boolean target on N=256.

In [11]:
from qos.theory import VariationalWarmstart
from qos.core.oracle_sketch import q_oracle_sketch_boolean

N = EXT_DIM
KF = 16  # fixed Fourier sparsity
M_vals_vw = [500, 1000, 2000, 5000, 10000]
vw_errors_m, baseline_errors_m = [], []

# Fixed random sparse boolean target (deterministic seed)
rng = jax.random.PRNGKey(42)
support = jax.random.choice(rng, N, shape=(KF,), replace=False)
f = jnp.zeros(N, dtype=jnp.int32).at[support].set(1)
target = jnp.exp(1j * jnp.pi * f)

for M_train in tqdm(M_vals_vw, desc='Training samples M'):
    # Variational warmstart
    vw = VariationalWarmstart(f, num_fourier_modes=KF, key=jax.random.PRNGKey(M_train))
    vw.fit(unit_num_samples=M_train)
    vw_diag = vw.predict()
    vw_errors_m.append(float(jnp.max(jnp.abs(vw_diag - target))))

    # Deterministic uniform baseline
    baseline_diag, _ = q_oracle_sketch_boolean(f, M_train)
    baseline_errors_m.append(float(jnp.max(jnp.abs(baseline_diag - target))))

    print(f'M={M_train}: baseline={baseline_errors_m[-1]:.4f}, variational={vw_errors_m[-1]:.4f}')

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(M_vals_vw, baseline_errors_m, 'o-', color='#888888', label='Uniform baseline (Zhao et al.)')
ax.plot(M_vals_vw, vw_errors_m, 's--', color=plt.get_cmap('Oranges')(0.75), label='Variational warmstart (Marena 2026)')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Training samples M'); ax.set_ylabel('Max oracle error')
ax.set_title(f'Variational Warmstart Sample Efficiency (N={N}, KF={KF})')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
if SAVE_FIGS: plt.savefig(f'{OUTPUT_DIR}/variational_warmstart.pdf', bbox_inches='tight')
if SHOW_FIGS: plt.show()
plt.close()


ModuleNotFoundError: No module named 'qos'

## Section 5.4: Interferometric Classical Shadow

In [12]:
from qos.theory import InterferometricClassicalShadow

key, subkey = random.split(key)
N = EXT_DIM
num_shadows_list = [50, 100, 200, 500, 1000]
weight_state = random.normal(subkey, (N,))
weight_state = weight_state / jnp.linalg.norm(weight_state)
key, subkey = random.split(key)
test_vecs = random.normal(subkey, (20, N))
test_vecs = test_vecs / jnp.linalg.norm(test_vecs, axis=1, keepdims=True)
shadow_errors = []

for ns in tqdm(num_shadows_list, desc='Shadow count'):
    shadow = InterferometricClassicalShadow(weight_state, num_shadows=ns)
    shadow.build_shadow()
    preds = shadow.predict(test_vecs)
    gt = jnp.einsum('d,td->t', weight_state, test_vecs)
    err = float(jnp.mean(jnp.abs(preds[:, 0] - jnp.real(gt))))
    shadow_errors.append(err)
    print(f'T={ns}: mean error = {err:.4f}')

fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(num_shadows_list, shadow_errors, 'o-', color=plt.get_cmap('Oranges')(0.75))
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Number of shadows T'); ax.set_ylabel('Mean prediction error')
ax.set_title('Interferometric Shadow Prediction Error')
ax.grid(True, which='both', alpha=0.3); plt.tight_layout()
if SAVE_FIGS: plt.savefig(f'{OUTPUT_DIR}/interferometric_shadow.pdf', bbox_inches='tight')
if SHOW_FIGS: plt.show()
plt.close()

ModuleNotFoundError: No module named 'qos'

## Appendix A: Depolarizing Noise Model

In [13]:
from qos.experiments.noise_benchmark import run_noise_benchmark
noise_results = run_noise_benchmark(dim=EXT_DIM, num_trials=EXT_TRIALS, output_dir=OUTPUT_DIR)
print('Noise benchmark complete. Keys:', list(noise_results.keys()))

ModuleNotFoundError: No module named 'qos'

## Appendix B: k-Forrelation Lower Bound Benchmark

In [14]:
from qos.experiments.forrelation_benchmark import run_forrelation_benchmark
forrelation_results = run_forrelation_benchmark(dim=EXT_DIM, num_trials=EXT_TRIALS, output_dir=OUTPUT_DIR)
print('Forrelation benchmark complete.')

ModuleNotFoundError: No module named 'qos'

## Appendix C: Kernel Shadow

In [15]:
from qos.experiments.kernel_benchmark import run_kernel_benchmark
kernel_results = run_kernel_benchmark(dim=EXT_DIM, num_trials=EXT_TRIALS, output_dir=OUTPUT_DIR)
print('Kernel benchmark complete.')

ModuleNotFoundError: No module named 'qos'

## Appendix D: Non-IID Scaling

In [16]:
from qos.experiments.non_iid_scaling import run_non_iid_scaling
noniid_results = run_non_iid_scaling(dim=EXT_DIM, num_trials=EXT_TRIALS, output_dir=OUTPUT_DIR)
print('Non-IID benchmark complete.')

ModuleNotFoundError: No module named 'qos'

## Summary: Improvement Factors

In [17]:
import glob
print('=' * 70)
print(f'BENCHMARK SUMMARY  (N={EXT_DIM}, FAST_MODE={FAST_MODE})')
print('=' * 70)

M_compare = M_vals[len(M_vals)//2]
K_test = max(1, EXT_DIM // 16)
idx = M_vals.index(M_compare)
u_err = uniform_errors[K_test][idx]
a_err = adaptive_errors[K_test][idx]
print(f'Adaptive oracle  (K=N/16, M={M_compare}):  uniform={u_err:.4f}  adaptive={a_err:.4f}')

if 16 in Q_vals and 2 in k_vals:
    z_s = zhao_samples[16]; h_s = hier_samples[16][2]
    print(f'Hierarchical     (Q=16, k=2):            Zhao={z_s:,}  hier={h_s:,}  ratio={z_s/h_s:.1f}x')

print()
print('PDFs saved to:', OUTPUT_DIR)
for f in sorted(glob.glob(f'{OUTPUT_DIR}/*.pdf')):
    print(f'  {f}')

BENCHMARK SUMMARY  (N=256, FAST_MODE=True)


NameError: name 'M_vals' is not defined